# Optimize And Benchmark

This notebook benchmarks two representative queries before and after `OPTIMIZE ... ZORDER BY`.

## Run Inputs
- `catalog`: optional override for the active catalog
- `schema`: schema that contains the fact tables and report outputs

## Source Tables
- `fact_order_items`
- `fact_orders`

## Output Tables
- `report_optimize_timings`
- `report_optimize_summary`

## Workflow
1. Define the benchmark queries.
2. Time each query before optimization.
3. Compute table statistics, run `OPTIMIZE`, and apply `ZORDER BY`.
4. Time the same queries again and persist the comparison tables.



## Capture Runtime Inputs

This benchmark is intentionally small and readable. Its goal is to document whether optimization helps the current sample, not to produce a production performance study.



In [ ]:
import time

from pyspark.sql import functions as F

dbutils.widgets.text("catalog", "")
dbutils.widgets.text("schema", "retailpulse")



## Resolve Context And Define Benchmark Queries

The queries represent two common access patterns: grouped item distribution and user-level basket aggregation.



In [ ]:
catalog = dbutils.widgets.get("catalog") or spark.sql("SELECT current_catalog()").first()[0]
schema = dbutils.widgets.get("schema") or spark.sql("SELECT current_schema()").first()[0]


def qname(name: str) -> str:
    return f"`{catalog}`.`{schema}`.`{name}`"


queries = {
    "department_hour_distribution": f"""
        SELECT department_id, order_hour_of_day, COUNT(*) AS items_seen
        FROM {qname("fact_order_items")}
        WHERE department_id IS NOT NULL
        GROUP BY department_id, order_hour_of_day
    """,
    "user_basket_summary": f"""
        SELECT user_id, AVG(basket_size) AS avg_basket_size
        FROM {qname("fact_orders")}
        GROUP BY user_id
    """,
}


# Collect the full result set so the timing measures execution work instead of only planning.
def time_query(sql_text: str) -> float:
    started = time.perf_counter()
    spark.sql(sql_text).collect()
    finished = time.perf_counter()
    return finished - started



## Time The Queries Before Optimization

Store every timing row so the final comparison table can be derived rather than hard-coded.



In [ ]:
timings = []
for query_name, query_sql in queries.items():
    timings.append((query_name, "before_optimize", time_query(query_sql)))



## Analyze And Optimize The Fact Tables

Statistics and layout changes are applied only after the baseline timings are collected.



In [ ]:
spark.sql(f"ANALYZE TABLE {qname('fact_order_items')} COMPUTE STATISTICS")
spark.sql(f"ANALYZE TABLE {qname('fact_orders')} COMPUTE STATISTICS")
spark.sql(f"OPTIMIZE {qname('fact_order_items')} ZORDER BY (product_id, department_id)")
spark.sql(f"OPTIMIZE {qname('fact_orders')} ZORDER BY (user_id)")



## Time The Queries After Optimization

Running the same SQL after optimization keeps the comparison apples-to-apples.



In [ ]:
for query_name, query_sql in queries.items():
    timings.append((query_name, "after_optimize", time_query(query_sql)))

timings_df = spark.createDataFrame(timings, ["query_name", "run_stage", "seconds"])

optimize_summary = (
    timings_df.groupBy("query_name")
    .pivot("run_stage", ["before_optimize", "after_optimize"])
    .agg(F.first("seconds"))
    .withColumn("seconds_saved", F.col("before_optimize") - F.col("after_optimize"))
    .withColumn(
        "speedup_ratio",
        F.when(F.col("after_optimize") > 0, F.col("before_optimize") / F.col("after_optimize"))
        .otherwise(F.lit(None)),
    )
)



## Persist And Review Benchmark Outputs

These report tables are reused by the dashboard and the report pack rather than recalculating the timings elsewhere.



In [ ]:
(
    timings_df.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(qname("report_optimize_timings"))
)

(
    optimize_summary.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(qname("report_optimize_summary"))
)

display(timings_df)
display(optimize_summary)



## Interpretation Note

Inspect the Databricks query profile for the benchmark queries above if you need richer evidence. Serverless notebooks do not expose the Spark UI directly, so this timing table is the canonical repo-tracked comparison.
